In [1]:
from pathlib import Path
import shutil

BASE = Path("/root/capsule/scratch/session_analysis_mlk")
BACKUP_ROOT = Path("/root/capsule/scratch/temp")
MOVS_FILENAME = "tongue_movs.parquet"


def backup_tongue_movs(
    base_dir: Path,
    backup_root: Path,
    *,
    overwrite: bool = False,
    progress: bool = True,
) -> dict:
    """
    Copy each session's tongue_movs.parquet to backup_root/[session]/tongue_movs.parquet.

    By default, refuses to overwrite existing backups (safer for re-runs).
    Set overwrite=True to force.
    """
    base_dir = Path(base_dir)
    backup_root = Path(backup_root)

    session_dirs = sorted(
        p for p in base_dir.glob("behavior_*") if (p / "intermediate_data").exists()
    )
    if not session_dirs:
        raise RuntimeError(f"No session folders with 'intermediate_data' found in {base_dir}")

    status = {}
    for sess_dir in session_dirs:
        sess_name = sess_dir.name
        src = sess_dir / "intermediate_data" / MOVS_FILENAME
        dst = backup_root / sess_name / MOVS_FILENAME

        try:
            if not src.exists():
                raise FileNotFoundError(f"source not found: {src}")

            if dst.exists() and not overwrite:
                if progress:
                    print(f"[skip] {sess_name}: backup already exists at {dst}")
                status[sess_name] = "skipped (exists)"
                continue

            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)  # copy2 preserves mtime

            # sanity check size matches
            if src.stat().st_size != dst.stat().st_size:
                raise RuntimeError(
                    f"size mismatch after copy: src={src.stat().st_size} dst={dst.stat().st_size}"
                )

            if progress:
                print(f"[ok] {sess_name}: backed up ({src.stat().st_size:,} bytes)")
            status[sess_name] = "ok"

        except Exception as e:
            if progress:
                print(f"[err] {sess_name}: {e!r}")
            status[sess_name] = f"error: {e!r}"

    return status


status = backup_tongue_movs(BASE, BACKUP_ROOT)
print("\nSummary:")
n_ok = sum(1 for v in status.values() if v == "ok")
n_skip = sum(1 for v in status.values() if v.startswith("skipped"))
n_err = sum(1 for v in status.values() if v.startswith("error"))
print(f"  ok: {n_ok}, skipped: {n_skip}, errors: {n_err}")
for k, v in status.items():
    if v != "ok":
        print(f"  {k}: {v}")

[ok] behavior_716325_2024-05-31_10-31-14: backed up (1,986,134 bytes)
[ok] behavior_717259_2024-06-28_11-17-19: backed up (1,846,224 bytes)
[ok] behavior_717263_2024-07-24_10-40-05: backed up (1,762,635 bytes)
[ok] behavior_751004_2024-12-20_13-26-07: backed up (1,766,805 bytes)
[ok] behavior_751004_2024-12-21_13-28-24: backed up (1,738,044 bytes)
[ok] behavior_751004_2024-12-22_13-09-11: backed up (2,103,374 bytes)
[ok] behavior_751004_2024-12-23_14-19-57: backed up (1,656,604 bytes)
[ok] behavior_751181_2025-02-25_12-12-30: backed up (1,074,391 bytes)
[ok] behavior_751181_2025-02-26_11-51-15: backed up (502,864 bytes)
[ok] behavior_751181_2025-02-27_11-24-44: backed up (462,426 bytes)
[ok] behavior_751766_2025-02-11_11-53-32: backed up (1,674,961 bytes)
[ok] behavior_751766_2025-02-13_11-31-21: backed up (1,813,426 bytes)
[ok] behavior_751766_2025-02-14_11-37-11: backed up (1,835,661 bytes)
[ok] behavior_751766_2025-02-15_12-08-11: backed up (1,985,604 bytes)
[ok] behavior_751769_202

In [4]:
from pathlib import Path
import traceback
import numpy as np
import pandas as pd

from aind_dynamic_foraging_behavior_video_analysis.ephys.tongue_ephys import load_intermediate_data


def compute_outbound_metrics(tongue_segmented: pd.DataFrame, jaw: pd.DataFrame) -> pd.DataFrame:
    """
    Compute outbound-only metrics per movement_id.

    Endpoint = frame with max Euclidean distance from mean jaw position.
    Outbound segment = frames from movement start up to and including endpoint.
    """
    required = {"movement_id", "time_in_session", "x", "y", "v"}
    missing = required - set(tongue_segmented.columns)
    if missing:
        raise ValueError("tongue_segmented missing columns: %s" % sorted(missing))
    if not {"x", "y"}.issubset(jaw.columns):
        raise ValueError("jaw must have columns ['x','y']")

    jx, jy = jaw[["x", "y"]].mean().astype(float)

    df = tongue_segmented.sort_values(["movement_id", "time_in_session"])

    rows = []
    for mid, grp in df.groupby("movement_id", sort=False):
        g = grp.dropna(subset=["x", "y"]).reset_index(drop=True)
        if g.empty:
            rows.append({
                "movement_id": mid,
                "out_duration": np.nan,
                "out_peak_velocity": np.nan,
                "out_mean_velocity": np.nan,
                "out_total_distance": np.nan,
            })
            continue

        # endpoint = farthest from jaw mean
        ed = np.sqrt((g["x"].to_numpy(float) - jx) ** 2 + (g["y"].to_numpy(float) - jy) ** 2)
        idx = int(np.argmax(ed))
        kept = g.iloc[: idx + 1]

        # outbound duration + distance (NaN if endpoint is the first frame)
        if len(kept) < 2:
            out_duration = np.nan
            out_total_distance = np.nan
        else:
            out_duration = float(kept["time_in_session"].iloc[-1] - kept["time_in_session"].iloc[0])
            x = kept["x"].to_numpy(float)
            y = kept["y"].to_numpy(float)
            out_total_distance = float(np.sqrt(np.diff(x) ** 2 + np.diff(y) ** 2).sum())

        # outbound velocity stats (nan-safe)
        v = kept["v"].to_numpy(float)
        if np.isfinite(v).any():
            out_peak_velocity = float(np.nanmax(v))
            out_mean_velocity = float(np.nanmean(v))
        else:
            out_peak_velocity = np.nan
            out_mean_velocity = np.nan

        rows.append({
            "movement_id": mid,
            "out_duration": out_duration,
            "out_peak_velocity": out_peak_velocity,
            "out_mean_velocity": out_mean_velocity,
            "out_total_distance": out_total_distance,
        })

    return pd.DataFrame(rows)


MOVS_FILENAME = "tongue_movs.parquet"


def augment_movs_with_outbound_in_place(
    base_dir: Path,
    *,
    progress: bool = True,
    dry_run: bool = False,
) -> dict:
    """
    For each session under base_dir, compute outbound-only metrics and REPLACE
    the original tongue_movs.parquet with an augmented version (original columns + out_*).

    Returns {session_name: status_string}.
    """
    base_dir = Path(base_dir)
    session_dirs = sorted(
        p for p in base_dir.glob("behavior_*") if (p / "intermediate_data").exists()
    )
    if not session_dirs:
        raise RuntimeError(f"No session folders with 'intermediate_data' found in {base_dir}")

    status = {}

    for sess_dir in session_dirs:
        sess_name = sess_dir.name
        try:
            if progress:
                print(f"==> {sess_name}: loading intermediate data")

            data = load_intermediate_data(sess_dir)
            jaw = pd.read_parquet(sess_dir / "intermediate_data" / "kps_raw_jaw.parquet")

            out_df = compute_outbound_metrics(data["kins"], jaw)

            movs = data["movs"]

            # Idempotency: drop any pre-existing out_* columns before merging
            out_cols = [c for c in movs.columns if c.startswith("out_")]
            if out_cols:
                if progress:
                    print(f"    dropping existing out_* columns: {out_cols}")
                movs = movs.drop(columns=out_cols)

            movs_aug = movs.merge(out_df, on="movement_id", how="left")
            nan_mask = movs_aug["out_duration"].isna()
            if nan_mask.any() and progress:
                nan_ids = set(movs_aug.loc[nan_mask, "movement_id"])
                ids_in_outdf = set(out_df["movement_id"])
                not_in_outdf = nan_ids - ids_in_outdf  # case 1
                in_outdf_all_nan = nan_ids & ids_in_outdf  # case 2 or 3

                print(f"    breakdown of {len(nan_ids)} NaN movements:")
                print(f"      missing from kins entirely: {len(not_in_outdf)}")
                print(f"      present in kins but no usable frames: {len(in_outdf_all_nan)}")

            if len(movs_aug) != len(movs):
                raise RuntimeError(
                    f"row count changed after merge: {len(movs)} -> {len(movs_aug)}"
                )

            n_missing = int(movs_aug["out_duration"].isna().sum())
            if progress and n_missing:
                print(f"    note: {n_missing}/{len(movs_aug)} movements missing out_ metrics")

            movs_path = sess_dir / "intermediate_data" / MOVS_FILENAME
            if not movs_path.exists():
                raise FileNotFoundError(f"expected movs parquet not found: {movs_path}")

            if dry_run:
                if progress:
                    print(f"[dry-run] would write {len(movs_aug)} rows to {movs_path}")
                status[sess_name] = f"dry-run ok ({len(movs_aug)} rows)"
            else:
                # Atomic write: temp file + replace
                tmp_path = movs_path.with_suffix(movs_path.suffix + ".tmp")
                movs_aug.to_parquet(tmp_path, index=False)
                tmp_path.replace(movs_path)
                if progress:
                    print(f"[ok] {sess_name}: wrote {len(movs_aug)} rows -> {movs_path.name}")
                status[sess_name] = f"ok ({len(movs_aug)} rows)"

        except Exception as e:
            if progress:
                print(f"[warn] {sess_name}: skipped -> {e!r}")
                traceback.print_exc()
            status[sess_name] = f"error: {e!r}"

    return status


# ---------------- Example usage ----------------
BASE = Path("/root/capsule/scratch/session_analysis_mlk")

# Recommend dry-run first:
status = augment_movs_with_outbound_in_place(BASE, dry_run=False)

# status = augment_movs_with_outbound_in_place(BASE, dry_run=False)
print("\nSummary:")
for k, v in status.items():
    print(f"  {k}: {v}")

==> behavior_716325_2024-05-31_10-31-14: loading intermediate data
    breakdown of 171 NaN movements:
      missing from kins entirely: 0
      present in kins but no usable frames: 171
    note: 171/7393 movements missing out_ metrics
[ok] behavior_716325_2024-05-31_10-31-14: wrote 7393 rows -> tongue_movs.parquet
==> behavior_717259_2024-06-28_11-17-19: loading intermediate data
    breakdown of 577 NaN movements:
      missing from kins entirely: 0
      present in kins but no usable frames: 577
    note: 577/6914 movements missing out_ metrics
[ok] behavior_717259_2024-06-28_11-17-19: wrote 6914 rows -> tongue_movs.parquet
==> behavior_717263_2024-07-24_10-40-05: loading intermediate data
    breakdown of 1130 NaN movements:
      missing from kins entirely: 0
      present in kins but no usable frames: 1130
    note: 1130/6650 movements missing out_ metrics
[ok] behavior_717263_2024-07-24_10-40-05: wrote 6650 rows -> tongue_movs.parquet
==> behavior_751004_2024-12-20_13-26-07: lo